<a href="https://colab.research.google.com/github/chidiview-ui/shiny-telegram/blob/main/Multi_agent_Brand_Monitoring_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install crewai
!pip install requests
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
import requests

class BrightDataWebSearchToolInput(BaseModel):
  """Input schema for BrightDataWebSearchTool."""
  title: str = Field(..., description="Topic of book to write about.")
class BrightDataWebSearchTool(BaseTool):
  name: str= " Web Search Tool"
  description: str = "Tool to Searche google and retreve the results."
  args_schema: Type[BaseModel] = BrightDataWebSearchToolInput

  def _run(self, title):
      host="brd.superproxy.io"
      port="33335"

      username= '<Get from brighdata.com in SERP API>'
      password= '<Get from brighdata.com in SERP API>'

      proxies = {
          'http': f'http://{username}:{password}@{host}:{port}',
          'https': f'http://{username}:{password}@{host}:{port}'
      }

      url = f"https://www.google.com/search?q={title}&brd_json=1&num=500"
      response = requests.get(url, proxies=proxies, verify=False)
      return response.json()["organic"]




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.5/195.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
def scrape_urls(input_urls: list[str], initial_params: dict, scraping_type: str):
  print(f"Scraping {scraping_type} for {len(input_urls)} urls")

  url = "https://api.brightdata.com.dataset/v3/trigger"

  headers = {
    "Authorization": f"Bearer {os.getenv('BRIGHTDATA_API_KEY')}",
    "Content-Type": "application/json"
  }

  data = [{"url": url} for url in input_urls]

  scraping_response = requests.post(url,
                                    headers=headers,
                                    params=initial_params,
                                    json=data)
  snapshot_id = scraping_response.json()["snapshot_id"]

  output_url = f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}"

  output_response = requests.get(output_url,
                                 headers=headers,
                                 params= {"format":"json"})
  return output_response.json()



In [3]:
!ollama pull deepseek-r1

/bin/bash: line 1: ollama: command not found


In [4]:
from crewai import LLM
llm = LLM(model="ollama/deepseek-r1")

In [ ]:
from crewai import Agent, Task
writer_agent = Agent(role = "Senior Report Writer Agent",
                       goal=""" Write a crisp bullet point report using
                                the analysis obtained from X analysis agent and how
                                the {brand name} is being used in the posts.""",
                     backstory = """You're an X post analysis expert with an
                                    understanding of the platform and its algorithms.
                                    You can write a crisp bullet point report using
                                    the analysis obtained from X analysis agent and how
                                    the {brand name} is being used in the posts.""")

writer_task = Task(description = """Write a crisp bullet point report about the
                                    analysis of the X posts and how the {brand name} is
                                    being used in the posts.""",
                    expected_output=""" A clear and concise bullet point report about the
                                        analysis of the X posts and how the {brand name} is
                                        being used in the posts. For each post in the input
                                        data, the output should be in the structured format
                                        provided to you:
                                         - A short title describing the {brand name}'s mention.
                                         - The url of the post.
                                         - Bullet points detailing the {brand name} mention
                                         in the post. Cover things like the tone of the post,
                                         the sentiment towards the brand, whether it drove
                                         engagement, whether it was a paid partnership, etc""",
                    agent=writer_agent)


In [ ]:
# Create a Flow

In [7]:
from crewai.flow import Flow, listen, start
from pydantic import BaseModel

# Define BrandMonitoringState
class BrandMonitoringState(BaseModel):
    brand_name: str = "nike"
    search_result: dict = {}


class BrandMonitoringFlow(Flow[BrandMonitoringState]):
     @start()
     def scrape_data(self):
         search_tool = BrightDataWebSearchTool()
         self.state.search_result = search_tool(self.state.brand_name)


     @listen(scrape_data)
     def scrape_platform_data_and_analyse(self):
      # X Scraping
      # NOTE: X_params is not defined.
      # NOTE: self.state.search_response should be self.state.search_result.
      # Also, self.state.search_result (from BrightDataWebSearchTool) is likely a list of dicts,
      # so accessing .x_urls directly will cause an AttributeError.
      # You need to parse the search_result to extract relevant X URLs.
      X_posts = scrape_urls(self.state.search_response.x_urls,
                            X_params, "X")
      # NOTE: XCrew is not defined.
      X_crew = XCrew().crew()
      X_response= X_crew.kickoff(Inputs={"X_data": X_posts,
                                        "brand_name": self.state.brand_name})

      #Youtube Scraping
      # NOTE: YouTube_params is not defined.
      # NOTE: self.state.search_response should be self.state.search_result.
      # Also, self.state.search_result (from BrightDataWebSearchTool) is likely a list of dicts,
      # so accessing .YouTube_urls directly will cause an AttributeError.
      # You need to parse the search_result to extract relevant YouTube URLs.
      YouTube_transcripts = scrape_urls(self.state.search_response.YouTube_urls,
                                  YouTube_params, "X")

      # NOTE: YouTubeCrew is not defined.
      YouTube_crew = YouTubeCrew().crew()
      # NOTE: YouTube_posts is not defined, should be YouTube_transcripts.
      YouTube_response= YouTube_crew.kickoff(Inputs={"YouTube_data": YouTube_posts,
                                        "brand_name": self.state.brand_name})